In [1]:
# Install once: pip install tiktoken
import tiktoken  # OpenAI-compatible token counting library

# Pick an encoding — cl100k_base matches many modern chat models
encoding = tiktoken.get_encoding("cl100k_base")  # Load the subword vocabulary table


In [2]:
sample_prompt = """
You are an annual report assistant for Acme Corp.
Answer ONLY from the context below.

=== CONTEXT START ===
Chunk 1 (source_id=acme_2023_p12): Revenue grew 19% year over year ...
=== CONTEXT END ===

Question: What was revenue growth in 2023?
"""

In [3]:
token_ids = encoding.encode(sample_prompt)  # Convert text to a list of token IDs
token_count = len(token_ids)  # Number of tokens in the whole prompt string

print(f"Words (split by space): {len(sample_prompt.split())}")  # Rough word count for comparison
print(f"Tokens (tiktoken): {token_count}") 

Words (split by space): 40
Tokens (tiktoken): 64


### Code for choosing the top chunks under budgets

In [3]:
import tiktoken

encoding=tiktoken.get_encoding("cl100k_base")

# Sample chunks returned from a vector database — newest first
retrieved_chunks = [
    {"source_id": "t23_p12", "page": 12, "text": "Revenue grew 19% year over year to $96.8B in 2023."},
    {"source_id": "t23_p8", "page": 8, "text": "Automotive gross margin improved sequentially in Q4."},
    {"source_id": "t23_p44", "page": 44, "text": "Energy storage deployments reached record levels."},
    {"source_id": "t22_p12", "page": 12, "text": "2022 revenue was $81.5B."},
]


MAX_CONTEXT_TOKENS = 600  # Designer-set budget for retrieval block only
INSTRUCTIONS = "Answer ONLY from CONTEXT. If missing, say you do not know."

def count_tokens(text):
    return len(encoding.encode(text))

def format_chunks(chunk):

    header=f"{chunk["source_id"]} p{chunk["page"]}"
    return f"{header}\n{chunk['page']}"

def build_context(chunks):
    parts=[]
    used=0
    for chunk in chunks:
        peice=format_chunks(chunk)
        peice_token=count_tokens(peice)
        if used+peice_token>MAX_CONTEXT_TOKENS:
            break
        parts.append(peice)
        used+=peice_token
    body='\n\n'.join(parts)
    return f"---context---\n{body}",used

context_blocks,ctx_tokens=build_context(retrieved_chunks)
question="what was 2023 revenue growth"
full_context=f"{INSTRUCTIONS}\nuser question:{question}\n{context_blocks}"

print("chunks included:",context_blocks.count("---")+1)
print("no of tokens : ",ctx_tokens)
print("Full Prompt : ",full_context)





chunks included: 3
no of tokens :  32
Full Prompt :  Answer ONLY from CONTEXT. If missing, say you do not know.
user question:what was 2023 revenue growth
---context---
t23_p12 p12
12

t23_p8 p8
8

t23_p44 p44
44

t22_p12 p12
12


In [11]:
max_history=4

# Full history saved on disk (could be hundreds)
full_history = [
    {"role": "user", "content": "Use only Acme annual report facts."},
    {"role": "assistant", "content": "Understood — grounded answers only."},
    {"role": "user", "content": "2022 revenue?"},
    {"role": "assistant", "content": "$81.5B per report."},
    {"role": "user", "content": "And in 2023?"},
    {"role": "assistant", "content": "19% growth to $96.8B."},
    {"role": "user", "content": "What was Q4 automotive margin trend?"},
]

def choosen_history(full_history,max_history):
    if len(full_history)<max_history:
        return full_history 
    dropped=full_history[:-max_history]
    kept=full_history[-max_history:]
    return kept,dropped

kept,dropped=choosen_history(full_history,max_history)
print(kept)
for msg in kept:
    print(msg["role"]," : ",msg["content"])

[{'role': 'assistant', 'content': '$81.5B per report.'}, {'role': 'user', 'content': 'And in 2023?'}, {'role': 'assistant', 'content': '19% growth to $96.8B.'}, {'role': 'user', 'content': 'What was Q4 automotive margin trend?'}]
assistant  :  $81.5B per report.
user  :  And in 2023?
assistant  :  19% growth to $96.8B.
user  :  What was Q4 automotive margin trend?


In [14]:
# pip install tiktoken
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

SAMPLE_PROMPT = """
You are a GreenCart FAQ assistant.
Answer ONLY from the context below.

=== CONTEXT START ===
Refunds are processed within 7 business days for eligible orders.
=== CONTEXT END ===

Question: What is the refund timeline?
"""

FULL_HISTORY = [
    {"role": "user", "content": "Use only GreenCart policy FAQ facts."},
    {"role": "assistant", "content": "Understood — grounded answers only."},
    {"role": "user", "content": "What is standard shipping time?"},
    {"role": "assistant", "content": "5-7 days in metro cities per FAQ."},
    {"role": "user", "content": "And refund timeline?"},
    {"role": "assistant", "content": "7 business days for eligible orders."},
    {"role": "user", "content": "What about warranty on electronics?"},
]

MAX_MESSAGES = 4


# TODO 1 — Write count_tokens(text) using encoding.encode(text)
def count_tokens(text: str) -> int:
    return len(encoding.encode(text))


# TODO 2 — Return (kept, dropped). If history fits, dropped = [].
#          kept = last max_messages items; dropped = everything before that.
def windowed_history(history: list, max_messages: int) -> tuple[list, list]:
    if len(history)<MAX_MESSAGES:
        return history
    dropped=history[:-MAX_MESSAGES]
    kept=history[-MAX_MESSAGES:]
    return kept,dropped




def main():
    word_count = len(SAMPLE_PROMPT.split())
    token_count = count_tokens(SAMPLE_PROMPT)

    print("=== Token check ===")
    print("Word count:", word_count)
    print("Token count:", token_count)

    kept, dropped = windowed_history(FULL_HISTORY, MAX_MESSAGES)

    print("\n=== Windowed history (max 4 messages) ===")
    print("Messages sent to model:", len(kept))
    print("Messages dropped:", len(dropped))
    if dropped:
        print("First dropped message:", dropped[0]["content"])


if __name__ == "__main__":
    main()


=== Token check ===
Word count: 36
Token count: 47

=== Windowed history (max 4 messages) ===
Messages sent to model: 4
Messages dropped: 3
First dropped message: Use only GreenCart policy FAQ facts.
